# LM basin sweep

In [1]:
import numpy as np
from anser import *
import os
os.chdir("/home/patrick/ansermodelling")
from models.eval import report_error_stats
from models import *
from models.train import pose_errors
import torch
import matplotlib.pyplot as plt
from data.reparametrisation import normal_to_angles, angles_to_normal
from models.eval import report_error_stats


In [2]:
coils_global = build_field_generator(N_turns,l,w,s,z_thick,centres,rotations)
f = lambda x : forward_model(x,coils_global)

## Positional Sweep

In [3]:
rng = np.random.default_rng(123)

In [4]:
test_set = np.load("data/test_set.npz")
measurements = test_set["xs"][:200]
poses = test_set["ys"][:200]

In [5]:
ds = np.linspace(0,0.15,20)

In [ ]:
poses_pred = np.empty((len(ds), len(poses), 6))
success = np.empty((len(ds), len(poses)), dtype=bool)
n_iters = np.empty((len(ds),len(poses)), dtype = int)

for i, d in enumerate(ds):
    perturbation = rng.normal(size=(len(poses), 3))
    perturbation /= np.linalg.norm(perturbation, axis=-1, keepdims=True)
    perturbation *= d
    pad = np.zeros((len(poses), 3))
    perturbation = np.concatenate([perturbation, pad], axis=-1)
    x0s = normal_to_angles(poses + perturbation)

    for j, x0 in enumerate(x0s):
        pose_pred, xs, suc = lm_solve(f, measurements[j], x0)
        poses_pred[i, j] = angles_to_normal(pose_pred)
        success[i, j] = suc
        n_iters[i,j] = len(xs)
    print(f"d = {d*1000:.1f} mm done")
    report_error_stats(poses_pred[i,:],poses,success[i,:])
    

d = 0.0 mm done
Mean pos error: 1.7e-15, mean angle error: 5.18e-15
Median pos error: 0, Median angle error: 3.95e-15
95% pos error : 9.86e-15 95% angle error: 1.36e-14
LM success rate: 0.355
Convergence rate: 1
Mean pos error of converged: 1.7e-15, mean angle error of converged: 5.18e-15 
d = 7.9 mm done
Mean pos error: 0.083, mean angle error: 0.316
Median pos error: 1.5e-11, Median angle error: 2.47e-12
95% pos error : 9.97e-10 95% angle error: 5.43e-10
LM success rate: 0.985
Convergence rate: 0.975
Mean pos error of converged: 2.49e-10, mean angle error of converged: 6.78e-11 
d = 15.8 mm done
Mean pos error: 0.995, mean angle error: 1.45
Median pos error: 2.37e-12, Median angle error: 9.24e-13
95% pos error : 5.76e-09 95% angle error: 2.48e-09
LM success rate: 0.975
Convergence rate: 0.965
Mean pos error of converged: 5.54e-10, mean angle error of converged: 1.7e-10 
d = 23.7 mm done
Mean pos error: 1.03, mean angle error: 1.66
Median pos error: 5.18e-12, Median angle error: 2.06e

In [ ]:
success.mean(axis = -1)

In [ ]:
exs = np.empty((len(ds), len(poses)))
ens = np.empty((len(ds), len(poses)))

true_t = torch.as_tensor(poses, dtype=torch.float64)

for i in range(len(ds)):
    pred_t = torch.as_tensor(poses_pred[i], dtype=torch.float64)
    ex, en = pose_errors(pred_t, true_t)
    exs[i], ens[i] = ex.numpy(), en.numpy()

converged = (exs < 1.0) & (ens < 1.0)
rate = converged.mean(axis=-1)

In [ ]:
rate

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ds * 1000, rate, marker="o", lw=1.5)
ax.axvline(8.77, color="r", ls="--", lw=1.2,
           label="Network mean position error")
ax.set_xlabel("Initial position error (mm)")
ax.set_ylabel("Convergence rate")
ax.set_ylim(0.7, 1.02)
ax.legend()
ax.grid(alpha=0.3)


In [ ]:
exs_log_476 = np.maximum(10**-12,exs[6,:])
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(np.log10(exs_log_476), bins = 50)
ax.axvline(1.0, color="k", ls="--", lw=1.0)
ax.text(1.1, ax.get_ylim()[1]*0.9, r"$\epsilon_p$", fontsize=10)
ax.set_xlabel("Log Position error (mm)")
ax.set_ylabel("Count")
plt.savefig("report/figs/pos_sweep_errors.png")
plt.show()

## Orientation sweep

In [ ]:
alphas = np.linspace(0, np.pi/2, 20)      # 0 to 90 degrees

poses_pred = np.empty((len(alphas), len(poses), 6))
n_true = poses[:,3:]
for i, a in enumerate(alphas):
    v = rng.normal(size=(len(poses), 3))
    v -= (v * n_true).sum(-1, keepdims=True) * n_true
    v /= np.linalg.norm(v, axis=-1, keepdims=True)
    n_pert = np.cos(a) * n_true + np.sin(a) * v

    x0s_6 = np.concatenate([poses[:, :3], n_pert], axis=-1)
    x0s = normal_to_angles(x0s_6)

    for j, x0 in enumerate(x0s):
        pose_pred, xs, _ = lm_solve(f, measurements[j], x0)
        poses_pred[i, j] = angles_to_normal(pose_pred)
    print(f"alpha = {np.degrees(a):.1f} deg done")
    report_error_stats(poses_pred[i,:],poses,success[i,:])


In [ ]:
exs = np.empty((len(ds), len(poses)))
ens = np.empty((len(ds), len(poses)))

true_t = torch.as_tensor(poses, dtype=torch.float64)

for i in range(len(ds)):
    pred_t = torch.as_tensor(poses_pred[i], dtype=torch.float64)
    ex, en = pose_errors(pred_t, true_t)
    exs[i], ens[i] = ex.numpy(), en.numpy()

converged = (exs < 1.0) & (ens < 1.0)
angles_rate = converged.mean(axis=-1)

In [ ]:

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(angles * 180/np.pi, angles_rate, marker="o", lw=1.5)
ax.axvline(6.2, color="r", ls="--", lw=1.2,
           label="Network mean angle error")
ax.set_xlabel("Initial angle error (deg)")
ax.set_ylabel("Convergence rate")
ax.set_ylim(0.7, 1.02)
ax.legend()
ax.grid(alpha=0.3)